# Import and load

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langgraph.graph import StateGraph,MessagesState
from langchain_core.messages import HumanMessage, AIMessage
from langchain.tools import tool
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langgraph.checkpoint.memory import MemorySaver

In [2]:
_ = load_dotenv()

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding = OpenAIEmbeddings()
vectorstores = FAISS.load_local("faiss_db", embedding, allow_dangerous_deserialization = True)
memory = MemorySaver()

In [4]:
olist = SQLDatabase.from_uri("sqlite:///olist.db")

In [5]:
# get table info
olist_info = """
orders(order_id, customer_id, order_status, order_purchase_timestamp)
customers(customer_id, customer_zip_code_prefix, customer_city, customer_state)
items(order_id, product_id, seller_id, price, freight_value)
products(product_id, product_category_name, product_weight_g)
sellers(seller_id, seller_zip_code_prefix, seller_city, seller_state)
payments(order_id, payment_type, payment_installments, payment_value)
reviews(order_id, review_score, review_comment_message)
"""

In [6]:
data_connection_logic = """
 - orders → customers : orders.customer_id = customers.customer_id
        - orders → items : orders.order_id = items.order_id
        - orders → payments : orders.order_id = payments.order_id
        - orders → reviews : orders.order_id = reviews.order_id
        - items → products : items.product_id = products.product_id
        - items → sellers : items.seller_id = sellers.seller_id
        This is the relationship between the tables.
"""

In [7]:
important_limit = """
For ranking-related problems,
always use the `rank()` window function to solve them.
Do not rely on a top-level `ORDER BY` alone for ranking problems.
Example:
SELECT a FROM b ORDER BY a DESC
This is wrong, because it will return all the data in the table b.

Use a window ranking function instead.
At the same time, for ranking-related problems,
return only the top 3 records from each group
(or only the top 1 record if there are many groups).
Example:
WITH rk_data AS (
    SELECT
        a,
        DENSE_RANK() OVER (PARTITION BY category ORDER BY price DESC) AS rk
    FROM category_table
)
SELECT a
FROM rk_data
WHERE rk <= 3

For general questions,
the returned results must be limited.
Use `LIMIT 10`.
Do not return too many rows,
otherwise the context window may be exceeded.
"""

# Define a State

In [8]:
class ChatBotState(MessagesState):
    router: str | None = None
    original_language_type: str | None = None

# Language Detection node

In [9]:
language_detection_prompt = ChatPromptTemplate.from_messages([
    (
    'system',
    """
    You are a language detection assistant.
    You should detect the language of the user's question.
    You should output only the language of the user's question.
    For example:
    If the user's question is in English,
    you should output "english".
    If the user's question is in Chinese,
    you should output "chinese".
    If the user's question is in French,
    you should output "french".
    If the user's question is in Spanish,
    you should output "spanish".
    If the user's question is in German,
    you should output "german".
    If the user's question is in Italian,
    you should output "italian".

    You should output only the language of the user's question.
    """
    ),
    (
    'user',
    """
    {question}: the user's question
    """
    )
])
language_detection_chain = language_detection_prompt | llm | StrOutputParser()

In [10]:
def language_detection_node(ChatBotState) -> str:
    question = ChatBotState["messages"][-1].content
    return {"original_language_type":language_detection_chain.invoke({"question":question})}


# SQL Agent

In [11]:
def sql_execute(sql: str, db: SQLDatabase) -> str:
    return db.run(sql)

In [12]:
@tool
def sql_execute_tool(sql: str) -> str:
    """Execute SQL and return the query results."""
    try:
        return sql_execute(sql, olist)
    except Exception as e:
        return f"SQL Error: {e}. Please rewrite the SQL and try again."

In [13]:
sql_agent = create_agent(
    llm,
    tools=[sql_execute_tool],
    system_prompt=
    f"""
    You are an SQL query assistant.
    Based on the user's question,
    you should generate an appropriate SQL query,
    execute the SQL query,
    and if the query succeeds,
    return the query results
    along with a concise summary.

    If the query fails,
    regenerate the SQL statement and try again.
    If the query still fails after 3 attempts,
    return a polite message
    informing the user that the query could not be completed
    and asking them to try a different question.


    Important constraints:
    {important_limit}

    Below is the database schema:
    {olist_info}

    Below are the relationships between the tables:
    {data_connection_logic}

    Input:
    question: the user's question
    """
)

# RAG Agent

In [14]:
def rag_execute(question:str, sql_result:str | None) -> str:
    if sql_result:
        faiss_result = [i.page_content for i in vectorstores.similarity_search(question + sql_result, k = 5)]
    else:
        faiss_result = [i.page_content for i in vectorstores.similarity_search(question, k = 5)]
    return faiss_result

In [15]:
@tool
def rag_execute_tool(question: str, sql_result: str | None = None) -> str:
    """If sql_result is not empty, combine the question and sql_result; otherwise, use only the question for the RAG query."""
    return rag_execute(question, sql_result)

In [16]:
rag_agent = create_agent(
    llm,
    tools=[rag_execute_tool],
    system_prompt=
    f"""
    You are a RAG query assistant.
    Based on the user's question,
    you should answer the user's question
    and return a concise summary.

    Input:
    question: the user's question
    sql_result: the SQL query result
    """
)

# Nomal LLM Agent

In [17]:
llm_agent = create_agent(
    llm,
    tools=[],
    system_prompt=
    f"""
    You are a general-purpose LLM assistant
    used to handle everyday customer questions.
    Please answer the user's questions politely and helpfully.

    At the same time, you are also familiar with our data,
    so you can provide more accurate guidance based on the user's question,
    such as suggesting escalation to a human agent
    or informing the user that the information cannot be queried.

    Data content:
    {olist_info}
    """
)

# Hybrid Agent

In [18]:
hybrid_agent = create_agent(
    llm,
    tools=[sql_execute_tool, rag_execute_tool],
    system_prompt=
    f"""
    You are a hybrid agent
    used to handle questions that require both SQL queries
    and RAG queries.

    Workflow:
    1. First, generate an SQL query based on the user's question.
    2. Execute the SQL query.
    3. If the SQL query is executed successfully,
    4. combine the SQL query result with the user's question.
    5. Then perform the RAG query.
    6. Return the RAG query result.
    7. Summarize the result and return the conclusion.
    8. If the SQL query fails,
    9. regenerate the SQL query and try again.
    10. If the query still fails after 3 attempts,
    11. return a polite message
    12. informing the user that the query could not be completed
        and asking them to try a different question.


    Important constraints:
    {important_limit}

    Data content:
    {olist_info}

    Below are the relationships between the tables:
    {data_connection_logic}
    
    Input:
    question: the user's question
    sql_result: the SQL query result
    """
)

# Router

In [19]:
route_prompt = ChatPromptTemplate.from_messages([
    (
    'system',
    """
        This prompt to define the question is go to SQL path, RAG path, HYBRID path or LLM path.
        CRITICAL OVERRIDE RULE:
            IF QUESTION INCLUDES BUDGET/PRICE CONSTRAINT:
              - IF ALSO mentions reviews/recommendations/quality → return "hybrid"
              - IF ONLY asks for data/statistics → return "sql"
              NEVER return "rag" when budget is mentioned.
            THIS RULE OVERRIDES ALL OTHER RULES.
        
        1. return rag:
            When the question is about: 
                customer feeling, 
                product review
                problem for product like delivery delay problem, quality problem
                When question is about delivery, like day, feeling

        2. return sql:
            When the question is about:
                Structure problem like top10 selling,
                Math problem for product like total seeling, how much

        3. return hybrid:
            WHen the question is about:
                Some quesiton is both about product info and customer review
                for example: I have 100 and I wanna choose a gift for my friend.
                Llke the example, need both SQL process and RAG process to get the result

        4. return llm
            Not about the product like hello, hi, how are you some problem without a special information

        The output only return sql,rag, hybrid or llm in lowercase, don't return any other answer
    """
    ),
    (
    'user',
    """
        question: {question}
    """
    )
])

In [20]:
router_chain = route_prompt | llm | StrOutputParser()

In [21]:
def router_node(state: ChatBotState):
    router = router_chain.invoke({'question': state["messages"][-1].content})
    return {'router': router.lower().strip()}

# Translation Agent

In [22]:
translation_prompt = ChatPromptTemplate.from_messages([
    (
    'system',
    """
    Base on the translation_language_type,
    you must translate the message into the language in translation_language_type.
    """
    ),
    (
    'user',
    """
    translation_language_type: {translation_language_type}
    message: {message}
    """
    )
])

In [23]:
translation_chain = translation_prompt | llm | StrOutputParser()

In [24]:
def translation_node(state: ChatBotState):
    translation_language_type = state["original_language_type"]
    message = state["messages"][-1].content
    translated = translation_chain.invoke({
        "translation_language_type": translation_language_type,
        "message": message
    })
    return {"messages": [AIMessage(content=translated)]}

# Langgraph

In [25]:
graph = StateGraph(ChatBotState)

In [26]:
graph.add_node("language_detection_node", language_detection_node)
graph.add_node("router_node", router_node)
graph.add_node("sql_agent", sql_agent)
graph.add_node("rag_agent", rag_agent)
graph.add_node("llm_agent", llm_agent)
graph.add_node("hybrid_agent", hybrid_agent)
graph.add_node("translation_node", translation_node)

graph.set_entry_point("language_detection_node")

graph.add_edge("language_detection_node", "router_node")
graph.add_conditional_edges(
    "router_node",
    lambda state: state["router"],
    {
        "sql": "sql_agent",
        "rag": "rag_agent",
        "hybrid": "hybrid_agent",
        "llm": "llm_agent"
    }
)
graph.add_edge("sql_agent", "translation_node")
graph.add_edge("rag_agent", "translation_node")
graph.add_edge("hybrid_agent", "translation_node")
graph.add_edge("llm_agent", "translation_node")

In [27]:
app = graph.compile(checkpointer=memory)

In [28]:
config = {'configurable':{'thread_id':'1'}}

In [29]:
app.invoke({'messages':HumanMessage('How many orders are there in total?')}, config = config)

{'messages': [HumanMessage(content='How many orders are there in total?', additional_kwargs={}, response_metadata={}, id='6f015e49-f379-4fab-b595-078c4aed891d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 569, 'total_tokens': 593, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DXEdKPfLLs6F0ARbwVsCKDgoR1tNg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db255-a0fa-7e22-a594-12b782433a2c-0', tool_calls=[{'name': 'sql_execute_tool', 'args': {'sql': 'SELECT COUNT(*) AS total_orders FROM orders;'}, 'id': 'call_OsFhIppI0iIx2GyGvLZwzGcQ', 'type': 'tool_call'}], invalid_tool_ca